In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
data = pd.read_csv("data_wrangling_part2.csv")

In [2]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161


This week is data modeling but for my project theres a few things I have to do: Come up with the best way to scrape the test from from the articles, analyze how many articles were lost in the process and redo the encoding for publisher size and checking the minimum article cutoff. Additionally I need to apply FinBert to get the sentiment analysis for the text of each article. Using the text instead of the headline makes sense because it is more representative of what the author is actually saying even at the cost of losing half of the data set due to paywalls and old articles not existing any more. Lastly, I need to determine the best model, how im splitting to handle time as well as being able to classify articles, and how im going to use the model to assign all the publishers a score. 

I used AI to help me make my article downloads (get_text_from_article function from last week) faster with using concurrent features and threading so it does not take 3 days to add the article text colummn. I am going to scrape all the articles, analyze how many rows actually have text, and go from there.  

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_text_from_article(url):
    try:
        article = Article(url)
        article.download(timeout=15)  
        article.parse()
        text = article.text
        return text if len(text) >= 1 else ""
    except Exception:
        return ""

results = [""] * len(data)

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {
        executor.submit(get_text_from_article, url): i
        for i, url in enumerate(data["url"])
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        results[idx] = future.result()

data["article_text"] = results

100%|██████████| 27357/27357 [00:00<00:00, 131121.58it/s]


In [4]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840,
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840,
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814,
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400,
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161,
